In [ ]:
import json
import os
import matplotlib.pyplot as plt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
pip install -U datasets


In [ ]:

%cd "/content/drive/MyDrive/NHERI_2025_SUMMER/"

/content/drive/MyDrive/NHERI_2025_SUMMER


In [ ]:
from transformers import ViTImageProcessor

model_name = "google/vit-base-patch16-224"
processor = ViTImageProcessor.from_pretrained(model_name)
processor

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ViTImageProcessor {
  "do_convert_rgb": null,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}

In [ ]:
from torchvision.transforms import (
    CenterCrop,
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    ToTensor,
    Resize,
)

image_mean, image_std = processor.image_mean, processor.image_std
size = processor.size["height"]

normalize = Normalize(mean=image_mean, std=image_std)

train_transforms = Compose(
    [
        RandomResizedCrop(size),
        RandomHorizontalFlip(),
        ToTensor(),
        normalize,
    ]
)
val_transforms = Compose(
    [
        Resize(size),
        CenterCrop(size),
        ToTensor(),
        normalize,
    ]
)
test_transforms = Compose(
    [
        Resize(size),
        CenterCrop(size),
        ToTensor(),
        normalize,
    ]
)

In [ ]:
from transformers import AutoImageProcessor

In [ ]:
image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
from transformers import ViTForImageClassification


In [ ]:
output_dir='./output-models-MODEL0/checkpoint-80/'
model_start= ViTForImageClassification.from_pretrained(output_dir, device_map="auto")

In [ ]:
output_dir='./checkpoint-74_Xiaoyu/'
model_walls=ViTForImageClassification.from_pretrained(output_dir, device_map="auto")

In [ ]:
output_dir= './checkpoint-510_Naomi/'
model_roof=ViTForImageClassification.from_pretrained(output_dir, device_map="auto")

In [ ]:
output_dir='./output-models_model_2/checkpoint-190/'
model_collapse=ViTForImageClassification.from_pretrained(output_dir, device_map="auto")

In [ ]:
%cd "/content/drive/MyDrive/NHERI_2025_SUMMER/"

/content/drive/MyDrive/NHERI_2025_SUMMER


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# Replace with the path to your image folder
image_folder = "./Framework_validation/"

# Get a list of image files in the folder
image_files = [f for f in os.listdir(image_folder) if f.endswith(('.jpg', '.jpeg', '.png', '.gif'))]


In [ ]:
image_files

['gettyimages-2174213668-612x612.jpg',
 'image (2).png',
 'istockphoto-1432276927-612x612.jpg',
 'istockphoto-178756342-612x612.jpg',
 '240_F_535599230_ovZygAw3AvEQImTVzCeJBwONHpSEGDvm.jpg',
 '240_F_866251216_VFYllNAdhvXPTxOOYY9ABE3phqKVgJYK.jpg',
 'download (1).jpg',
 'image (3).png',
 'istockphoto-174749922-612x612.jpg',
 '240_F_970355776_wMZfZ8AfycOVvMGlUkND0GFXBckMzspC.jpg',
 '240_F_986748352_kcSpmO2Wud9tMoj1cAgpu67GeCgnM2wr.jpg',
 '240_F_1064057366_US2caOEciSYNzwiu0WMCWnRdVnV9U1Yv.jpg',
 '240_F_1213614472_ZyhrFUHf8EIyPTMHwbqghy9AQG64TIIN.jpg',
 '240_F_817140365_ZA0bguWe3rs6qJ3gO17drsyS5taPeyZC.jpg',
 '240_F_321937465_05azthkeyV2bTsydSM1XzEqayYEqwdEm.jpg',
 '240_F_617495558_orCpMYaDx5f6NYiV8or89mb4huZandoh.jpg',
 '240_F_1079550821_Qe0aSwi1r3MN9CZW4ezYqudDfkSTVIAx.jpg',
 '240_F_218023995_wx1AwqUoP0kMIDl3ZXLUShRghSiwl4XY.jpg',
 '240_F_1263824027_1IqvHKhfVPnnBfKeQC7AOfagPOQtNHdM.jpg',
 '240_F_697433913_APBX6ypQIB2msV7FRTcKjrkh2sI4evlT.jpg',
 '240_F_608387204_WUJ6Y5NbacisPVEsOhaAGv6NBB

In [ ]:
import torch

In [ ]:
total_images=[]
for image_file in image_files:

    image_path = os.path.join(image_folder, image_file)
    print(image_path)
    img = Image.open(image_path)
    inputs = image_processor(img, return_tensors="pt")
    Logits=[]
    for model in [model_start, model_walls, model_roof, model_collapse]:
      inputs = inputs.to(model.device)
      outputs_for_model=model(**inputs)
      logits_for_model=outputs_for_model.logits
      probs, indices = torch.topk(logits_for_model.softmax(dim=1), k=2)
      Logits.append(probs.detach())
    total_images.append(Logits)


./Framework_validation/gettyimages-2174213668-612x612.jpg
./Framework_validation/image (2).png
./Framework_validation/istockphoto-1432276927-612x612.jpg
./Framework_validation/istockphoto-178756342-612x612.jpg
./Framework_validation/240_F_535599230_ovZygAw3AvEQImTVzCeJBwONHpSEGDvm.jpg
./Framework_validation/240_F_866251216_VFYllNAdhvXPTxOOYY9ABE3phqKVgJYK.jpg
./Framework_validation/download (1).jpg
./Framework_validation/image (3).png
./Framework_validation/istockphoto-174749922-612x612.jpg
./Framework_validation/240_F_970355776_wMZfZ8AfycOVvMGlUkND0GFXBckMzspC.jpg
./Framework_validation/240_F_986748352_kcSpmO2Wud9tMoj1cAgpu67GeCgnM2wr.jpg
./Framework_validation/240_F_1064057366_US2caOEciSYNzwiu0WMCWnRdVnV9U1Yv.jpg
./Framework_validation/240_F_1213614472_ZyhrFUHf8EIyPTMHwbqghy9AQG64TIIN.jpg
./Framework_validation/240_F_817140365_ZA0bguWe3rs6qJ3gO17drsyS5taPeyZC.jpg
./Framework_validation/240_F_321937465_05azthkeyV2bTsydSM1XzEqayYEqwdEm.jpg
./Framework_validation/240_F_617495558_orCpMYa

In [ ]:
i=4
total_images[i]

[tensor([[0.9818, 0.0182]]),
 tensor([[0.5296, 0.4704]]),
 tensor([[0.7248, 0.2752]]),
 tensor([[0.7165, 0.2835]])]

In [ ]:
Total_images=np.array(total_images)

In [ ]:
for i in range(len(Total_images)):
  image_probs=Total_images[i].squeeze()
  max_values = image_probs.max(axis=1,keepdims=True)
  indices_of_max = np.where(image_probs == max_values)[1]
  if indices_of_max[0]==0:
    damage_type_start=0
  else:
    damage_type_start=1
  if indices_of_max[1]==1:
    damage_type_roof=0
  else:
    damage_type_roof=1
  if indices_of_max[2]==0:
    damage_type_wall=0
  else:
    damage_type_wall=1
  if indices_of_max[3]==0:
    damage_type_collapsed=0
  else:
    damage_type_collapsed=1

  vector_start=[]
  if damage_type_start==0:
    vector_start.append(image_probs[0,indices_of_max[0]] )
    if damage_type_collapsed==0:
      vector_start.append(image_probs[3,indices_of_max[3]])
      if damage_type_wall==1 or damage_type_roof==1:
        vector_start.append(image_probs[1,indices_of_max[1]])
        vector_start.append(image_probs[2,indices_of_max[2]])
        print('Category IV')
      if damage_type_wall==0 and damage_type_roof==0:
        vector_start.append(image_probs[1,indices_of_max[1]])
        vector_start.append(image_probs[2,indices_of_max[2]])
        print('Category III')
    else:
      vector_start.append(image_probs[3,indices_of_max[3]])
      print('Category V')
  else:
    vector_start.append(image_probs[0,indices_of_max[0]])
    if damage_type_roof==1 or damage_type_wall==1:
      vector_start.append(image_probs[1,indices_of_max[1]])
      vector_start.append(image_probs[2,indices_of_max[2]])
      print('Category II')
    if damage_type_roof==0 and damage_type_wall==0:
      vector_start.append(image_probs[1,indices_of_max[1]])
      vector_start.append(image_probs[2,indices_of_max[2]])
      print('Category I')
  print(returnOverallProb(vector_start))



Category IV
0.99999815
Category IV
0.9999858
Category IV
1.0
Category IV
0.99998724
Category IV
0.999331
Category IV
0.99979836
Category IV
0.9999362
Category IV
0.9999987
Category IV
0.9991322
Category IV
0.9999168
Category IV
0.9972674
Category IV
0.9860606
Category IV
0.99943936
Category IV
0.9991858
Category IV
0.9996112
Category IV
0.9995041
Category IV
0.99866605
Category IV
0.99899983
Category IV
0.9992875
Category IV
0.9992064
Category IV
0.9999994
Category IV
0.99866736
Category IV
0.9991545
Category IV
0.9999903
Category IV
0.9999903
Category IV
0.99983275


In [ ]:
vector_start

[np.float32(0.98177725), np.float32(0.59080523)]

np.float32(0.99254334)

In [ ]:
import numpy as np
import itertools
import math

In [ ]:
def returnOverallProb(probList):
    probList1 = np.asarray(probList)
    probList0 = 1-probList1
    overallProb = 0
    num = len(probList)
    indexList = range(1,num+1)
    indexListE = range(0,num+1)
    #qn depends on num
    if num>1 or num==1:
        qn = 1
    #print ('%-10s %-10s %-5s %-5s %-5s %-5s'% ('combs','sum(c)/num','pZ1','pZ0','pCc','product'))
    #print ('-'*50)
    for i in indexListE:
        combs = np.asarray(list(itertools.combinations(indexList,i))) #List of combinations created from the selection of i elements from the list

        for sampleCombs in combs:
            #run through all sampleCombs
            #productTemp = Π(p(Ci=ci|xi))  #xi = Prior ; if we believe that the predicition is the same as the ground truth, what is the product
            # (c1|x1) ---- > Ci is the prediction and ci is the value of the predition
            #
            productTemp = 1
            weightAdjust = 0
            for j in indexList:
                if j in sampleCombs:
                    #productTemp = productTemp*p(Ci=1|xi)
                    productTemp = productTemp*probList1[j-1]
                    if probList1[j-1]>=0.5:
                        weightAdjust = 1
                else:
                    #productTemp = productTemp*p(Ci=0|xi)
                    productTemp = productTemp*probList0[j-1] #if the xi is not in combination
            #probC = p(C=c|Ci=ci,...)
            probC_z1 = max(weightAdjust,len(sampleCombs)/num) #posterior probability for the whole combination
            #probC_z1 = math.ceil(len(sampleCombs)/num)
            #print (probC_z1)
            #if insufficient, trustworthy is 0.5
            probC_z0 = max(math.ceil(len(sampleCombs)/num),0.5) #prior probability for the whole combination
            probC = probC_z1*qn+probC_z0*(1-qn)
            productTemp = productTemp*probC
            #print ('%-10s %-10.3f %-5.1f %-5.1f %-5.1f %-5f'% (sampleCombs,len(sampleCombs)/num,probC_z1,probC_z0,probC,productTemp))
            #overallProb = overallProb+Π(p(Ci=ci|xi))
            overallProb = overallProb+productTemp
    #print ('-'*50)
    return overallProb

In [ ]:
Logits

[tensor([[0.6408, 0.3592]], grad_fn=<TopkBackward0>),
 tensor([[0.9633, 0.0367]], grad_fn=<TopkBackward0>),
 tensor([[0.6923, 0.3077]], grad_fn=<TopkBackward0>),
 tensor([[0.8660, 0.1340]], grad_fn=<TopkBackward0>)]